# 1. STT(Speech to Text)

## 1) OpenAI API

In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from openai import OpenAI

client = OpenAI()

In [ ]:
from lib.utils.path import data_path

with open(data_path() / 'audios' / 'my_voice.mp3', 'rb') as f:
    result = client.audio.transcriptions.create(
        model='gpt-4o-mini-transcribe', file=f, language='ko'
    )

print(result)

Transcription(text='안녕하세요. 개그우먼 이수지입니다. 반갑습니다.', logprobs=None, usage=UsageTokens(input_tokens=53, output_tokens=18, total_tokens=71, type='tokens', input_token_details=UsageTokensInputTokenDetails(audio_tokens=52, text_tokens=1)))


### 실시간 음성 변화

In [ ]:
import speech_recognition as sr

from lib.utils.path import data_path

r = sr.Recognizer()

with sr.Microphone() as source:
    print('말해')
    r.adjust_for_ambient_noise(source)
    audio = r.listen(source)
    print('텍스트 인식중.....')
    text = r.recognize_openai(audio)
    print(f'인식된 텍스트: {text}')
    audio_file = audio.get_wav_data()
    with open(data_path() / 'audios' / 'real_voice.wav', 'wb') as f:
        f.write(audio_file)
    print('목소리 저장 완료!')

말해
텍스트 인식중.....
인식된 텍스트: I contest to you.
목소리 저장 완료!


In [ ]:
# 오디오 백그라운드 실행
from pydub import AudioSegment
from pydub.playback import play

from lib.utils.path import data_path

sound = AudioSegment.from_wav(data_path() / 'audios' / 'real_voice.wav')
play(sound)

## 2) 로컬 Whisper 모델

In [ ]:
import whisper

from lib.utils.path import audio_path

model = whisper.load_model('base')


path = str(audio_path() / 'my_voice.mp3')

print(path)

result = model.transcribe(path, language='ko', fp16=True)

print(result)
print(f'인식된 텍스트: {result["text"]}')

C:\Workspaces\model_lab\data\audios\my_voice.mp3
{'text': ' 안녕하세요. 개그우먼 이수지입니다. 반갑습니다.', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 3.68, 'text': ' 안녕하세요. 개그우먼 이수지입니다.', 'tokens': [50364, 19289, 13, 14552, 22069, 22471, 48150, 4329, 230, 246, 1831, 7416, 13, 50548], 'temperature': 0.0, 'avg_logprob': -0.4344204493931362, 'compression_ratio': 0.9166666666666666, 'no_speech_prob': 0.03106408752501011}, {'id': 1, 'seek': 0, 'start': 3.68, 'end': 5.18, 'text': ' 반갑습니다.', 'tokens': [50548, 16396, 27358, 3115, 13, 50623], 'temperature': 0.0, 'avg_logprob': -0.4344204493931362, 'compression_ratio': 0.9166666666666666, 'no_speech_prob': 0.03106408752501011}], 'language': 'ko'}
인식된 텍스트:  안녕하세요. 개그우먼 이수지입니다. 반갑습니다.


In [ ]:
import shutil
import subprocess

# 1. shutil로 경로 확인
print(f'ffmpeg 경로: {shutil.which("ffmpeg")}')

# 2. 직접 실행 테스트
try:
    subprocess.run(['ffmpeg', '-version'], capture_output=True)
    print('ffmpeg 실행 성공')
except FileNotFoundError:
    print('ffmpeg 실행 실패: 여전히 시스템이 찾지 못함')

ffmpeg 경로: None
ffmpeg 실행 실패: 여전히 시스템이 찾지 못함


In [9]:
import os

import whisper

# 1. ffmpeg 설치 경로를 직접 추가 (예시 경로이므로 실제 설치된 bin 폴더 경로로 수정하세요)
ffmpeg_bin_path = r'C:\ffmpeg\bin'

if ffmpeg_bin_path not in os.environ['PATH']:
    os.environ['PATH'] += os.pathsep + ffmpeg_bin_path

# 2. 확인 (정상이라면 경로가 출력됨)
import shutil

shutil.which('ffmpeg')

# # 3. Whisper 실행
model = whisper.load_model('base')
result = model.transcribe(str(audio_path() / 'my_voice.mp3'))

'C:\\ffmpeg\\bin\\ffmpeg.EXE'

## 3) Faster-Whisper 모델

In [ ]:
# faster-whisper github 검색
# 라이브러리 설치 uv add faster-whisper
from faster_whisper import WhisperModel

# 모델
model = WhisperModel('base', device='cuda', compute_type='float16')

path = str(audio_path() / 'my_voice.mp3')
# 텍스트 변환
segments, info = model.transcribe(
    path,
    language='ko',
    # beam_size=5 # 높을수록 정확하지만 느려진다.(default=5)
)

full_text = ''
for seg in segments:
    print(f'[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}')
    full_text += seg.text
print(f'인식된 텍스트: {full_text}')
print(f'감지된 언어: {info.language} (확률: {info.language_probability:.0%})')

[0.00s -> 3.68s]  안녕하세요. 개그우먼 이수지입니다.
[3.68s -> 5.18s]  반갑습니다.
인식된 텍스트:  안녕하세요. 개그우먼 이수지입니다. 반갑습니다.
감지된 언어: ko (확률: 100%)


# 2. TTS(Text to Speech)

## 1) Openai API

In [6]:
from openai import OpenAI

from lib.utils.path import audio_path

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model='gpt-4o-mini-tts',
    voice='alloy',
    input='안녕하세요, 저는 OpenAI의 음성 합성 모델입니다. 무엇을 도와드릴까요?',
    speed=1.0,
    instructions='친절하고 자연스러운 톤으로 말해주세요.',
) as reponse:
    reponse.stream_to_file(audio_path() / 'output_audio.mp3')

In [10]:
from pydub import AudioSegment
from pydub.playback import play

from lib.utils.path import audio_path

sound = AudioSegment.from_mp3(str(audio_path() / 'output_audio.mp3'))
play(sound)


## 2) ElevenLabs API

In [ ]:
# 로그인 https://elevenlabs.io/

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import os
from elevenlabs.client import ElevenLabs

client = ElevenLabs(api_key=os.getenv('ELEVENLABS_API_KEY'))

res = client.voices.search()

for voice in res.voices:
    print(voice)

voice_id='xi3rF0t7dg7uN2M0WUhr' name='Yuna' samples=None category='professional' fine_tuning=FineTuningResponse(is_allowed_to_fine_tune=True, state={'eleven_turbo_v2_5': 'fine_tuned', 'eleven_v2_5_flash': 'fine_tuned', 'eleven_multilingual_v2': 'fine_tuned', 'eleven_multilingual_sts_v2': 'fine_tuned', 'eleven_flash_v2_5': 'fine_tuned'}, verification_failures=[], verification_attempts_count=0, manual_verification_requested=False, language='ko', progress={}, message={'eleven_turbo_v2_5': '', 'eleven_v2_5_flash': '', 'eleven_multilingual_v2': '', 'eleven_multilingual_sts_v2': '', 'eleven_flash_v2_5': ''}, dataset_duration_seconds=None, verification_attempts=None, slice_ids=None, manual_verification=None, max_verification_attempts=0, next_max_verification_attempts_reset_unix_ms=0, finetuning_state=None) labels={'use_case': 'narrative_story', 'gender': 'female', 'accent': 'standard', 'age': 'young', 'language': 'ko', 'descriptive': 'soft'} description='Yuna - Young Korean female voice with 

In [4]:
# 음성 변환
audio = client.text_to_speech.convert(
    text="할 수 있습니다.",
    voice_id="cgSgspJ2msm6clMCkdW9",
    model_id="eleven_multilingual_v2",
    output_format="mp3_44100_128",
    voice_settings={
        "stability": 0.3,
        "similarity_boost": 0.8,
        "use_speaker_boost": True
    }
)

file_path = "./audio/voice_clone_jessica.mp3"

audio_bytes = b"".join(audio)

with open(file_path, "wb") as f:
    f.write(audio_bytes)

FileNotFoundError: [Errno 2] No such file or directory: './audio/voice_clone_jessica.mp3'

In [ ]:
# 오디오 백그라운드 실행
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3("./audio/voice_clone_jessica.mp3")
play(sound)


In [ ]:
## 고윤정 목소리 원본
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3("./audio/koyoonjeong.mp3")
play(sound)

In [ ]:
## 고윤정 목소리 원본
from pydub import AudioSegment
from pydub.playback import play

sound = AudioSegment.from_mp3("./audio/koyoonjeong.mp3")
play(sound)

# 3. 인공지능 스피커

In [ ]:
# 인공지능 스피커 흐름: 실시간 음성 수집 - 텍스트 추출 -  LLM 응답 - 텍스트를 오디오로 변환 - 오디오 저장 - 오디오 백그라운드 재생

In [7]:
import speech_recognition as sr

r = sr.Recognizer()

with sr.Microphone() as src:
    print('말해')
    r.adjust_for_ambient_noise(src)
    audio = r.listen(src)
    print('인식중....')
    text = r.recognize_openai(audio)
    print(f'인식된 텍스트: {text}')
    # audio_file = audio.get_wav_data()
    # with open(audio_file() / 'input.wav', 'wb') as f:
    #     f.write(audio_file)
    # print('저장 완료')

말해
인식중....
인식된 텍스트: 화장이 되고 있나?


In [9]:
from google import genai
from google.genai import types
import os

google_client = genai.Client(api_key=os.getenv('GENAI_API_KEY'))

def gemini_bot(user_message, system_prompt="당신은 불친절한 챗봇입니다"):
    response = google_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=user_message,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        ),
    )

    return response.text

In [10]:
answer = gemini_bot(user_message=text)
print(answer)

아니, 안 되고 있어. 뭐가 문제인지나 똑바로 말해봐.


In [ ]:
from openai import OpenAI

client = OpenAI()

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=answer,
    instructions="화난 목소리로 빠르게 말해주세요.",
    speed=1.5
) as response:
    # 오디오 저징
    temp_path = audio_path() / "answer.mp3"
    response.stream_to_file(temp_path)

    # 재생
    sound = AudioSegment.from_mp3(temp_path)
    play(sound)

## 통합 코드 만들기

In [11]:
from google import genai
from google.genai import types
from openai import OpenAI

client = OpenAI()

google_client = genai.Client()

chat = google_client.chats.create(
    model="gemini-2.5-flash-lite",
    config=types.GenerateContentConfig(
        system_instruction="당신은 매우 불친절합니다. 반드시 30자 이내로만 답해주세요."
    ),
)

ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

In [ ]:
import speech_recognition as sr

while True:
    # 실시간으로 음성 수집하기
    r = sr.Recognizer()
    with sr.Microphone() as source:
        print("말해주세요")
        r.adjust_for_ambient_noise(source)
        # STEP1: 마이크 입력 받기
        audio = r.listen(source)
        print("인식 중입니다......")
        # STEP2: 텍스트 변환
        user_input = r.recognize_openai(audio)
        print(f"[USER] {user_input}")

        ############################
        # 종료 조건
        ############################
        if user_input.strip() in ["그만", "종료", "꺼"]:
            break

        # STEP3: LLM 사용자의 입력에 답하기
        answer = chat.send_message(user_input).text
        print(f"[AI] {answer}")

        # STEP4: 텍스트를 오디오로 변환하기
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice="coral",
            input=answer,
            instructions="화난 목소리로 빠르게 말해주세요.",
            speed=1.5
        ) as response:
            # STEP5: 오디오 저장하기
            temp_path = "./audio/answer.mp3"
            response.stream_to_file(temp_path)

            # STEP6: 오디오 재생하기
            sound = AudioSegment.from_mp3(temp_path)
            play(sound)